In [ ]:
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Cargar el dataset final
df = pd.read_csv('dataset_final_colab.csv')

# 2. Definir la función de ventaneo
def crear_ventanas(data, window_size=60):
    X, y = [], []
    # Buscamos el índice de Ciudad Bolívar (el objetivo)
    target_idx = df.columns.get_loc('ciudad_bolivar')
    data_array = data.values

    for i in range(window_size, len(data_array)):
        X.append(data_array[i-window_size:i, :]) # Bloque de 60 días
        y.append(data_array[i, target_idx])      # El valor del día 61
    return np.array(X), np.array(y)

# Creamos las ventanas (X tendrá 27 columnas de entrada)
X, y = crear_ventanas(df, window_size=60)

# 3. Dividir en Entrenamiento (80%) y Prueba (20%)
# Importante: No usamos 'shuffle' porque el orden del tiempo importa
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Datos listos. Forma de X_train: {X_train.shape}")
# Debería decir algo como (X, 60, 27)

In [ ]:
# 4. Construir la arquitectura LSTM
model = Sequential([
    LSTM(100, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1) # Predicción final
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 5. Configurar avisos para mejorar el entrenamiento
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)
]

# 6. ¡ENTRENAR!
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt # El dibujante de las gráficas

# 1. Hacer las predicciones
predictions = model.predict(X_test)

# 2. Usar el scaler_y para volver a metros reales
# Cargamos el escalador que guardaste
sc_y = joblib.load('scaler_y_final.pkl')
y_test_real = sc_y.inverse_transform(y_test.reshape(-1, 1))
predictions_real = sc_y.inverse_transform(predictions)

# 3. Graficar
plt.figure(figsize=(15, 6))
plt.plot(y_test_real, label='Nivel Real (Orinoco)', color='blue', alpha=0.7)
plt.plot(predictions_real, label='Predicción IA', color='red', linestyle='--')
plt.title('Comparativa: Nivel Real vs Predicción del Modelo')
plt.xlabel('Días (Set de Prueba)')
plt.ylabel('Metros / Nivel')
plt.legend()
plt.show()

print("✅ Gráfica generada. Si la línea roja sigue a la azul, ¡tienes un modelo ganador!")

In [ ]:
#test de prueba
# 1. Hacer las predicciones con el set de prueba
predictions = model.predict(X_test)

# 2. Cargar el escalador Y para volver a metros reales
sc_y = joblib.load('scaler_y_final.pkl')

# 3. Des-normalizar (Invertir el escalado de 0-1 a Metros)
y_test_real = sc_y.inverse_transform(y_test.reshape(-1, 1))
predictions_real = sc_y.inverse_transform(predictions)

# 4. Crear la gráfica comparativa
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 8))
plt.plot(y_test_real, label='Nivel Real (Histórico)', color='#1f77b4', linewidth=2)
plt.plot(predictions_real, label='Predicción IA (LSTM)', color='#ff7f0e', linestyle='--', linewidth=2)

plt.title('Validación del Modelo: Niveles del Río Orinoco', fontsize=16)
plt.xlabel('Días del periodo de prueba', fontsize=12)
plt.ylabel('Nivel del Río (Metros)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 5. Calcular el error en metros reales para que lo entiendas mejor
error_promedio_metros = np.mean(np.abs(y_test_real - predictions_real))
print(f"✅ El error promedio de tu IA es de: {error_promedio_metros:.3f} metros.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import joblib

# 1. Ajuste de índices dinámico
total_test = len(X_test)
# Tomamos una ventana que esté un poco antes del final para tener 7 días reales que comparar
# Usaremos la penúltima semana disponible en el set de test
indice_inicio_ventana = total_test - 14

if indice_inicio_ventana < 0:
    print("⚠️ El set de test es muy pequeño. Usando el primer dato disponible.")
    indice_inicio_ventana = 0

ventana_test = X_test[indice_inicio_ventana : indice_inicio_ventana + 1]
# Los 7 días reales que siguen a esa ventana
valores_reales_siguiente_semana = y_test[indice_inicio_ventana : indice_inicio_ventana + 7]

# 2. Predicción Recursiva de 7 días
predicciones_7_dias = []
ventana_actual = ventana_test.copy()

for i in range(7):
    pred_norm = model.predict(ventana_actual, verbose=0)
    predicciones_7_dias.append(pred_norm[0, 0])

    nueva_fila = ventana_actual[0, -1, :].copy()
    target_idx = df.columns.get_loc('ciudad_bolivar')
    nueva_fila[target_idx] = pred_norm[0, 0]

    nueva_ventana = np.append(ventana_actual[0, 1:, :], nueva_fila.reshape(1, 27), axis=0)
    ventana_actual = nueva_ventana.reshape(1, 60, 27)

# 3. Des-normalizar
sc_y = joblib.load('scaler_y_final.pkl')
predicciones_metros = sc_y.inverse_transform(np.array(predicciones_7_dias).reshape(-1, 1))
reales_metros = sc_y.inverse_transform(valores_reales_siguiente_semana.reshape(-1, 1))

# 4. Graficar
plt.figure(figsize=(12, 6))
plt.plot(range(1, 8), reales_metros, label='Valor Real', marker='o', color='#1f77b4')
plt.plot(range(1, 8), predicciones_metros, label='Predicción IA', marker='s', linestyle='--', color='#ff7f0e')

plt.title('Validación de Pronóstico: Real vs IA (7 Días)')
plt.xlabel('Día del pronóstico')
plt.ylabel('Metros')
plt.legend()
plt.grid(True, alpha=0.3)

# Mostrar errores
for i, (real, pred) in enumerate(zip(reales_metros, predicciones_metros)):
    error = abs(real - pred)
    plt.text(i+1, pred, f'{error[0]:.2f}m', ha='center', va='bottom')

plt.show()

print(f"📊 Error medio en esta semana: {np.mean(np.abs(reales_metros - predicciones_metros)):.3f} m")